# 03 - Assign Cluster Labels to ALL Trajectories  (Stage 3)

**Purpose.** Load the chosen k-means model **and the saved StandardScaler**, and
assign a `cluster_label` to every trajectory by nearest-centroid lookup
(`predict`, no iteration). The scaler from notebook 02 must be applied to the raw
features before `predict`, so every trajectory is standardized with the *same*
means/stds the model was trained on.

- `complete` and `partial` trajectories are labelled (for `partial`, the last
  known position was already used as the proxy for the missing later day(s) in
  Stage 1).
- `early_loss` trajectories have no usable features -> `cluster_label = -1`.

If `config.GROUP_MAP` is set, a merged `cluster_group` column is also added;
otherwise `cluster_group` mirrors `cluster_label`.

**Input.** `data/features.parquet`, `data/kmeans_models/kmeans_k{BEST_K}.pkl`,
`data/kmeans_models/scaler.pkl`.
**Output.** `data/labeled_trajectories.parquet`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)
print("sampling days:", C.DAYS, "| feature space:", C.FEATURE_SPACE)

project root: /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_50_150_stdZ
sampling days: [30, 50, 100, 150] | feature space: zscore


## 3.1  Choose k and load the model + scaler

In [2]:
BEST_K = 40          # <-- set after inspecting notebook 02 (papermill: -p BEST_K <k>)

In [3]:
# Parameters
BEST_K = 50


In [4]:
import pickle
with open(C.MODELS_DIR / f"kmeans_k{BEST_K}.pkl", "rb") as f:
    km = pickle.load(f)
with open(C.MODELS_DIR / "scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
print("loaded model with", km.n_clusters, "clusters and the shared StandardScaler")

loaded model with 50 clusters and the shared StandardScaler


## 3.2  Predict labels for every labellable trajectory

In [5]:
features = pd.read_parquet(C.FEATURES_FILE)
labelable = features.status != "early_loss"
# Standardize with the SAME scaler, then apply the SAME per-day weighting the
# model was fit with (config.DAY_WEIGHTS via pipeline.feature_weight_vector()).
# The k-means centroids live in this weighted space, so predict MUST see it too;
# omitting * W would assign labels in a different space than the model was fit in.
W = P.feature_weight_vector()
X_all = scaler.transform(P.build_feature_matrix(features[labelable])) * W
labels = np.full(len(features), -1, dtype=np.int32)
labels[labelable.to_numpy()] = km.predict(X_all).astype(np.int32)
features["cluster_label"] = labels
print(features.cluster_label.value_counts().sort_index().to_string())

cluster_label
-1      120000
 0      888233
 1      257513
 2      231156
 3      514756
 4      107422
 5      225007
 6      135618
 7      222327
 8       70637
 9     1557718
 10     178373
 11     635700
 12     137311
 13     221944
 14      95828
 15     293593
 16     300226
 17     188027
 18     203683
 19     263836
 20     238078
 21     160134
 22    2443951
 23     553667
 24     308210
 25     213369
 26     106383
 27     680744
 28     848793
 29     207835
 30      60147
 31     203301
 32     254816
 33     595826
 34     118872
 35     219612
 36      59102
 37     134890
 38      51992
 39     126225
 40     134680
 41     328730
 42     160231
 43      52062
 44     235502
 45     279899
 46     241299
 47     101868
 48     113441
 49     197433


## 3.2b  Merge into pathway groups (optional)

If `config.GROUP_MAP` is non-empty, merge the raw clusters into groups; the
result is stored in `cluster_group`. With an empty map, `cluster_group` simply
equals `cluster_label`.

In [6]:
if C.GROUP_MAP:
    groups, raw2grp = P.apply_group_map(features.cluster_label.to_numpy(), C.GROUP_MAP, BEST_K)
    features["cluster_group"] = groups.astype(np.int32)
    print("raw cluster -> group:", raw2grp)
    print(features.loc[features.cluster_group >= 0, "cluster_group"]
          .value_counts().sort_index().to_string())
else:
    features["cluster_group"] = features["cluster_label"]
    print("GROUP_MAP empty -> cluster_group mirrors cluster_label")

raw cluster -> group: {0: 1, 1: 5, 2: 4, 3: 1, 4: 2, 5: 3, 6: 1, 7: 0, 8: 1, 9: 5, 10: 3, 11: 1, 12: 3, 13: 1, 14: 5, 15: 1, 16: 5, 17: 0, 18: 4, 19: 3, 20: 2, 21: 1, 22: 1, 23: 4, 24: 1, 25: 5, 26: 5, 27: 5, 28: 1, 29: 1, 30: 3, 31: 0, 32: 1, 33: 0, 34: 1, 35: 4, 36: 1, 37: 2, 38: 5, 39: 0, 40: 6, 41: 7, 42: 8, 43: 9, 44: 10, 45: 11, 46: 12, 47: 13, 48: 14, 49: 15}


cluster_group
0     1335706
1     7162194
2      480390
3      864674
4     1208118
5     3263773
6      134680
7      328730
8      160231
9       52062
10     235502
11     279899
12     241299
13     101868
14     113441
15     197433


## 3.3  Save labelled trajectories

In [7]:
features.to_parquet(C.LABELED_FILE, index=False)
print("saved", features.shape, "->", C.LABELED_FILE)

saved (16280000, 16) -> /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_50_150_stdZ/data/labeled_trajectories.parquet


## 3.4  Summary

In [8]:
lbl = features[features.cluster_label >= 0]
print(f"Labelled {len(lbl):,} trajectories into {BEST_K} clusters "
      f"({(features.cluster_label==-1).sum():,} early_loss left unlabelled).")
print(f"Saved to {C.LABELED_FILE}")

Labelled 16,160,000 trajectories into 50 clusters (120,000 early_loss left unlabelled).
Saved to /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_50_150_stdZ/data/labeled_trajectories.parquet
